# FP_NB_Pipeline_Logger
**Northbrook Lending: run logging for `FP_PL_Daily_Load`**

Called as the **final step** of the pipeline, on both the success path and every failure path.
Appends one row per pipeline run to **`FP_Gold.dbo.ETL_RunLog`**:

| Column | Source |
|---|---|
| `RunId`, `PipelineName`, `StartTimeUtc` | pipeline system variables (`@pipeline().RunId`, `@pipeline().TriggerTime`) |
| `Status`, `FailedStep`, `ErrorMessage` | set by the path that called the logger (success, or the step that failed) |
| `SilverRowsLoaded`, `GoldFactRepaymentRows`, `GoldFactLoanSnapshotRows` | exit values returned by the Silver and Gold notebooks |
| `DQChecks`, `DQPass`, `DQWarn`, `DQFail` | exit value returned by `FP_NB_DataQuality` |

The log feeds the *Data Quality & Operations* page of the Power BI report.

In [ ]:
# Parameters - FP_PL_Daily_Load passes in the run details and each notebook's exit value
run_id = ""
pipeline_name = "FP_PL_Daily_Load"
trigger_time = ""         # when the pipeline was triggered (ISO text)
status = "Succeeded" # overall result - Succeeded or Failed
failed_step = ""          # name of the step that failed, if any
error_message = ""        # error text from the failed step
silver_result = ""          # JSON exit value from FP_NB_Silver_Transform
gold_result = ""            # JSON exit value from FP_NB_Gold_Model
dq_result = ""              # JSON exit value from FP_NB_DataQuality

In [ ]:
# json reads the notebook results, uuid makes run IDs, datetime handles times
import json, uuid, datetime as dt
# Spark data types - used to define the layout of the run log table
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType, LongType

# Find the OneLake root folder of a lakehouse by name, in whichever workspace this runs
def lakehouse_root(name: str) -> str:
    try:
        # Preferred - ask Fabric for this lakehouse by name
        lh = notebookutils.lakehouse.get(name)
        # The lookup can return a dict or an object - read its properties either way
        props = lh["properties"] if isinstance(lh, dict) else lh.properties
        # Take its OneLake Tables path and cut off /Tables to leave the lakehouse root
        return props["oneLakeTablesPath"].rstrip("/").rsplit("/Tables", 1)[0]
    except Exception:
        # Fallback if the lookup fails - build the OneLake path from the current workspace name
        ws = notebookutils.runtime.context["currentWorkspaceName"]
        return f"abfss://{ws}@onelake.dfs.fabric.microsoft.com/{name}.Lakehouse"

# Read a notebook's JSON exit value safely
def parse(s):
    """Notebook exit values arrive as JSON strings; tolerate blanks and junk."""
    try:
        # Read the JSON into a dict; return an empty dict if the value is blank or not valid JSON
        return json.loads(s) if s else {}
    except Exception:
        return {}

# Turn the pipeline's ISO timestamp text into a datetime, dropping the Z and fractions of a second; None if blank or invalid
def ts(s):
    try:
        return dt.datetime.fromisoformat(s.replace("Z", "").split(".")[0]) if s else None
    except Exception:
        return None

# Read the summaries passed back by the Silver, Gold and DQ notebooks
silver, gold, dq = parse(silver_result), parse(gold_result), parse(dq_result)
# End time is now; start time is when the pipeline was triggered
end_ts, start_ts = dt.datetime.utcnow(), ts(trigger_time)
# Row count per Gold table, or an empty dict if Gold didn't report any
gold_tables = gold.get("tables", {}) if isinstance(gold.get("tables"), dict) else {}

# Build the single log row for this pipeline run
row = (
    # Run ID (a random one if none was passed in), pipeline name, status, start and end times
    run_id or str(uuid.uuid4()), pipeline_name, status, start_ts, end_ts,
    # Run length in seconds, if the start time is known
    float((end_ts - start_ts).total_seconds()) if start_ts else None,
    # Failed step and error message, if any; the message is cut to 2,000 characters
    failed_step or None, (error_message or "")[:2000] or None,
    # Total rows loaded to Silver across all tables
    sum(t.get("silver_rows", 0) for t in silver.get("tables", [])) if silver else None,
    # Row counts for the two Gold fact tables
    gold_tables.get("FactRepayment"), gold_tables.get("FactLoanSnapshot"),
    # Data quality check counts - total, pass, warn and fail
    dq.get("checks"), dq.get("PASS"), dq.get("WARN"), dq.get("FAIL"),
)
# Column names and types for the run log table
schema = StructType([
    StructField("RunId", StringType()), StructField("PipelineName", StringType()), StructField("Status", StringType()),
    StructField("StartTimeUtc", TimestampType()), StructField("EndTimeUtc", TimestampType()),
    StructField("DurationSeconds", DoubleType()), StructField("FailedStep", StringType()),
    StructField("ErrorMessage", StringType()), StructField("SilverRowsLoaded", LongType()),
    StructField("GoldFactRepaymentRows", LongType()), StructField("GoldFactLoanSnapshotRows", LongType()),
    StructField("DQChecks", LongType()), StructField("DQPass", LongType()),
    StructField("DQWarn", LongType()), StructField("DQFail", LongType())])

# Turn the row into a one-row Spark table
log = spark.createDataFrame([row], schema)
# Append the row to ETL_RunLog in Gold, building up a history of every pipeline run
log.write.format("delta").mode("append").option("mergeSchema", "true") \
   .save(f"{lakehouse_root('FP_Gold')}/Tables/dbo/ETL_RunLog")
# Show the row that was logged
display(log)

In [ ]:
# End the notebook and tell the pipeline which run was logged and its status
notebookutils.notebook.exit(json.dumps({"notebook": "FP_NB_Pipeline_Logger", "logged_run": row[0], "status": status}))